# README Examples Validation

This notebook contains all code examples from the README.md file.
Run this notebook to verify that:
1. All examples execute without errors
2. Outputs match what's documented in the README

**After making changes to blox**, run this notebook and update the README if outputs have changed.

In [1]:
# Setup: ensure blox is importable and configure devices.
import sys

sys.path.insert(0, '../src')

# Set up 4 CPU devices for sharding examples.
# Must be done before JAX is imported.
import chex

chex.set_n_cpu_devices(4)

## Your First Layer

From README section: "Your First Layer"

In [2]:
import blox as bx
import jax
import jax.numpy as jnp


class Linear(bx.Module):

  def __init__(self, graph: bx.Graph, output_size: int, rng: bx.Rng):
    super().__init__(graph)
    self.output_size = output_size
    self.rng = rng

  def __call__(self, params: bx.Params, x: jax.Array):
    # Parameters are created lazily on first use.
    # No need to specify input shapes upfront or preallocate memory!
    kernel, params = self.get_param(
        params,
        name='kernel',
        shape=(x.shape[-1], self.output_size),
        init=jax.nn.initializers.glorot_uniform(),
        rng=self.rng,
    )
    bias, params = self.get_param(
        params,
        name='bias',
        shape=(self.output_size,),
        init=jax.nn.initializers.zeros,
        rng=self.rng,
    )
    return x @ kernel + bias, params


print('Linear defined successfully!')

Linear defined successfully!


## Composing Layers

From README section: "Composing Layers"

In [3]:
class MLP(bx.Module):

  def __init__(
      self,
      graph: bx.Graph,
      hidden_size: int,
      output_size: int,
      rng: bx.Rng,
  ):
    super().__init__(graph)
    # graph.child('name') creates a unique path for parameters
    self.hidden = Linear(graph.child('hidden'), hidden_size, rng=rng)
    self.output = Linear(graph.child('output'), output_size, rng=rng)

  def __call__(self, params: bx.Params, x: jax.Array):
    x, params = self.hidden(params, x)
    x = jax.nn.relu(x)
    return self.output(params, x)


print('MLP defined successfully!')

MLP defined successfully!


## Initialization & Inspection

From README section: "Initialization & Inspection"

**Important**: The output of `bx.display()` should match the README. If it differs, update the README!

In [4]:
# Define the structure.
graph = bx.Graph('net')
rng = bx.Rng(graph.child('rng'))
model = MLP(graph.child('mlp'), hidden_size=128, output_size=10, rng=rng)

# Initialize the parameter container and initialize the RNG state (seed).
# We need the RNG to initialize parameters so we initialize it first.
params = bx.Params()
params = rng.seed(params, seed=42)

# Run a forward pass to trigger lazy parameter initialization.
dummy_input = jnp.ones((1, 784))
_, params = model(params, dummy_input)

# Lock it down to prevent accidental parameter creation during training.
params = params.locked()

# Visualize the full graph and parameter structure.
bx.display(graph, params)

net: Graph # Params: 101772 (397.6 KB)(
  rng=Rng # Params: 2 (12 B)(
    __init__=Rng(auto_fold_in_axes=True), # Repeated python obj at 0x75dea8384500
    seed=Param[N](
      shape=(),
      dtype='key<fry>',
      metadata={'tag': 'rng_seed'},
      value=# jax.Array key<fry>()
        Array((), dtype=key<fry>) overlaying:
        [ 0 42]
      ,
    ),
    counter=Param[N](shape=(), dtype='uint32', metadata={'tag': 'rng_counter'}, value=<jax.Array(4, dtype=uint32)>),
  ),
  mlp=MLP # Params: 101770 (397.5 KB)(
    __init__=MLP(hidden_size=128, output_size=10, rng=Rng(auto_fold_in_axes=True)),
    hidden=Linear # Params: 100480 (392.5 KB)(__init__=Linear(output_size=128, rng=Rng(auto_fold_in_axes=True)), kernel=Param[T](shape=(784, 128), dtype='float32', value=<jax.Array float32(784, 128)- too large to summarize.>), bias=Param[T](shape=(128,), dtype='float32', value=<jax.Array float32(128,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:128>)),
    output=Linear # Params: 1290 (5.0 KB)(__init__=Linear(output_size=10, rng=Rng(auto_fold_in_axes=True)), kernel=Param[T](shape=(128, 10), dtype='float32', value=<jax.Array float32(128, 10) ≈0.00068 ±0.12 [≥-0.21, ≤0.21] nonzero:1_280>), bias=Param[T](shape=(10,), dtype='float32', value=<jax.Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)>)),
  ),
)

**Expected output structure** (values may vary due to random initialization):
- `rng` and `mlp` should be siblings under `net`
- `hidden` and `output` should be nested inside `mlp`
- For the multi-graph example, `rng` appears under `net2` since it was defined there

<cell_type>markdown</cell_type>## Parallel Execution (vmap & shard_map)

From README section: "Batching & Parallel RNG"

**Note**: In vmap/shard_map, all lanes get the same RNG key by default. To get unique randomness per lane, you must manually fold in the axis index.

In [ ]:
# Create a fresh graph for this example.
graph2 = bx.Graph('dropout_net')
dropout_rng = bx.Rng(graph2.child('dropout_rng'))
dropout = bx.Dropout(graph2.child('dropout'), rate=0.5, rng=dropout_rng)

# Initialize params.
params2 = bx.Params()
params2 = dropout_rng.seed(params2, seed=42)

# Initialize dropout.
dummy_input = jnp.ones((4, 8))
_, params2 = dropout(params2, dummy_input, is_training=True)
params2 = params2.locked()

# === Approach 1: Pass batch index explicitly ===
# The simplest way to understand RNG folding is to pass the index manually.


def apply_with_explicit_index(params, inputs, batch_idx):
  # Fold in the batch index to get a unique seed for this lane.
  original_seed = dropout_rng.get_seed(params)
  folded_seed = jax.random.fold_in(original_seed, batch_idx)
  params = dropout_rng.seed(params, seed=folded_seed)

  # Use RNG (now produces unique values for this batch element).
  outputs, params = dropout(params, inputs, is_training=True)

  # Restore original seed (required for replicated params).
  params = dropout_rng.seed(params, seed=original_seed)
  return outputs, params


# Create batch indices manually.
batch_size = 4
batch_indices = jnp.arange(batch_size)
inputs_batch = jnp.ones((batch_size, 8))

# vmap over both inputs and the explicit index.
batched_outputs_v1, _ = jax.vmap(
    apply_with_explicit_index, in_axes=(None, 0, 0), out_axes=(0, None)
)(params2, inputs_batch, batch_indices)

print('Approach 1: Explicit batch index')
print('Each batch element has different dropout mask:')
for i in range(4):
  print(f'  Batch {i}: {jnp.sum(batched_outputs_v1[i] != 0).item()} non-zero')

In [ ]:
# === Approach 2: Use jax.lax.axis_index ===
# When using axis_name with vmap, JAX can provide the index automatically.
# This is cleaner when you don't want to thread the index through your code.


def apply_with_axis_index(params, inputs):
  # jax.lax.axis_index returns the lane index for the named axis.
  original_seed = dropout_rng.get_seed(params)
  folded_seed = jax.random.fold_in(
      original_seed, jax.lax.axis_index('batch')
  )
  params = dropout_rng.seed(params, seed=folded_seed)

  outputs, params = dropout(params, inputs, is_training=True)

  params = dropout_rng.seed(params, seed=original_seed)
  return outputs, params


# axis_name='batch' is required for jax.lax.axis_index to work.
batched_outputs_v2, _ = jax.vmap(
    apply_with_axis_index,
    in_axes=(None, 0),
    out_axes=(0, None),
    axis_name='batch',
)(params2, inputs_batch)

print('\nApproach 2: jax.lax.axis_index')
print('Same result, but index is implicit:')
for i in range(4):
  print(f'  Batch {i}: {jnp.sum(batched_outputs_v2[i] != 0).item()} non-zero')

## Parameter Metadata & Sharding

From README section: "Parameter Metadata & Sharding"

In [6]:
from jax.sharding import NamedSharding
from jax.sharding import PartitionSpec as P

graph3 = bx.Graph('net')
rng3 = bx.Rng(graph3.child('rng'))
linear = bx.Linear(
    graph3.child('linear'),
    output_size=1024,
    rng=rng3,
    kernel_metadata={'sharding': (None, 'model')},
    bias_metadata={'sharding': ('model',)},
)


# Define an initialization function.
def init(x):
  params = rng3.seed(bx.Params(), seed=42)
  _, params = linear(params, x)
  return params.locked()


# Abstract evaluation to get the Params structure (no memory allocation).
inputs3 = jnp.ones((4, 4))
abstract_params = jax.eval_shape(init, inputs3)

# Create the sharding specification from metadata.
mesh = jax.make_mesh((4,), ('model',))

params_sharding = jax.tree.map(
    lambda p: NamedSharding(mesh, P(*p.sharding)),
    abstract_params,
    is_leaf=lambda x: isinstance(x, bx.Param),
)

# JIT-compile the init function with out_shardings.
# Params are created directly on the correct devices, with no memory overhead.
sharded_init = jax.jit(init, out_shardings=params_sharding)
sharded_params = sharded_init(inputs3)


@jax.jit(in_shardings=(params_sharding, None), donate_argnames='params')
def forward(params, x):
  return linear(params, x)


out, new_params = forward(sharded_params, inputs3)

print('Output shape:', out.shape)
print('Sharding example completed successfully!')

/tmp/ipykernel_36583/1864585531.py:27: DeprecationWarning: The default axis_types will change in JAX v0.9.0 to jax.sharding.AxisType.Explicit. To maintain the old behavior, pass `axis_types=(jax.sharding.AxisType.Auto,) * len(axis_names)`. To opt-into the new behavior, pass `axis_types=(jax.sharding.AxisType.Explicit,) * len(axis_names)
  mesh = jax.make_mesh((4,), ('model',))


Output shape: (4, 1024)
Sharding example completed successfully!


## Recurrence & Scanning

From README section: "Recurrence & Scanning"

In [7]:
# Create a fresh graph for LSTM example.
graph4 = bx.Graph('lstm_net')
rng4 = bx.Rng(graph4.child('rng'))
lstm = bx.LSTM(graph4.child('lstm'), hidden_size=128, rng=rng4)
params4 = rng4.seed(bx.Params(), seed=42)

# Create sequence input [Batch, Time, Features].
inputs_sequence = jnp.ones((2, 10, 64))  # 2 batches, 10 timesteps, 64 features

# Initialize the LSTM state.
state, params4 = lstm.initial_state(
    params4, inputs_sequence[:, 0, :]
)  # Use single timestep for init

# Run efficient compiled scan over a sequence [Batch, Time, Features].
# It automatically handles carry propagation.
(outputs, final_state), params4 = lstm.apply(
    params4, inputs_sequence, prev_state=state
)

print('LSTM outputs shape:', outputs.shape)
print('Final hidden state shape:', final_state.hidden.shape)
print('Final cell state shape:', final_state.cell.shape)

LSTM outputs shape: (2, 10, 128)
Final hidden state shape: (2, 128)
Final cell state shape: (2, 128)


## Training (JIT & Gradients)

From README section: "Training (JIT & Gradients)"

In [8]:
# Use the MLP model from above.
# Re-create to have fresh params.
graph5 = bx.Graph('train_net')
rng5 = bx.Rng(graph5.child('rng'))
model5 = MLP(graph5.child('mlp'), hidden_size=32, output_size=1, rng=rng5)
params5 = rng5.seed(bx.Params(), seed=42)

# Initialize.
train_inputs = jnp.ones((8, 10))  # 8 samples, 10 features
_, params5 = model5(params5, train_inputs)
params5 = params5.locked()


@jax.jit(donate_argnames='params')
def train_step(params, inputs, targets):
  # Split params into two sets.
  # Trainable: weights, biases (we want gradients for these).
  # Non-trainable: Rng, batch stats, EMA (we just want the updated values).
  trainable, non_trainable = params.split()

  def loss_fn(t, nt):
    # Merge parameters to run the forward pass.
    predictions, new_params = model5(t.merge(nt), inputs)

    # Calculate the loss.
    loss = jnp.mean((predictions - targets) ** 2)

    # Extract the updated non-trainable state to pass it out.
    _, new_non_trainable = new_params.split()
    return loss, new_non_trainable

  # Calculate gradients and capture the auxiliary state (non_trainable updates).
  grads, new_non_trainable = jax.grad(loss_fn, has_aux=True)(
      trainable, non_trainable
  )

  # Update the trainable weights using SGD.
  new_trainable = jax.tree.map(lambda w, g: w - 0.01 * g, trainable, grads)

  # Merge the updated weights with the updated non-trainable state.
  return new_trainable.merge(new_non_trainable)


# Run a few training steps.
targets = jnp.zeros((8, 1))  # Target outputs
for step in range(5):
  params5 = train_step(params5, train_inputs, targets)

# Verify training worked by checking loss decreased.
predictions, _ = model5(params5, train_inputs)
final_loss = jnp.mean((predictions - targets) ** 2)
print(f'Final loss after 5 steps: {final_loss:.6f}')
print('Training example completed successfully!')

Final loss after 5 steps: 0.002036
Training example completed successfully!
